# Prepare Test Data for NILM Inference

This notebook prepares test data from `Electricity_P.csv` exactly the way the preprocessing notebook did, and packages it ready to send to the backend.

## How it works

The preprocessing notebook split the data **chronologically**:

| Split | Range |
|-------|-------|
| Train | First 80% |
| Val   | 80% → 85% |
| **Test** | **Last 15%** ← used here |

For each appliance it then:
1. Detected ON/OFF transitions in the appliance column
2. Extracted each ON event with lead/trail context from the **AGGREGATE only**
3. Built the **7-channel tensor**
4. Padded to `MAX_LEN`

This notebook does the same thing for the **TEST split only**, and produces a list of event dicts ready to send to the backend one by one.

## 1. Imports

In [1]:
import os
import numpy as np
import pandas as pd

## 2. Constants

> ⚠️ These **must match** the preprocessing notebook exactly.

In [2]:
MAINS_COL = 'Whole-House Meter'

APPLIANCE_NAMES = {
    'CDE': 'Clothes Dryer',
    'CWE': 'Clothes Washer',
    'DWE': 'Dishwasher',
    'FGE': 'Kitchen Fridge',
    'HPE': 'Heat Pump',
    'TVE': 'TV',
    'WHE': 'Whole-House Meter',
}

TARGET_APPLIANCES = [
    'Clothes Dryer',
    'Clothes Washer',
    'Dishwasher',
    'Kitchen Fridge',
    'Heat Pump',
    'TV',
]

WATTS_ON_THRESHOLD = 50.0

CONTEXT_CONFIG = {
    'Clothes Dryer' : {'lead': 30, 'trail': 30},
    'Clothes Washer': {'lead': 5,  'trail': 15},
    'Dishwasher'    : {'lead': 30, 'trail': 30},
    'Heat Pump'     : {'lead': 60, 'trail': 60},
    'Kitchen Fridge': {'lead': 10, 'trail': 10},
    'TV'            : {'lead': 2,  'trail': 2},
}

MIN_EVENT_DURATION = {
    'Clothes Dryer' : 10,
    'Clothes Washer': 10,
    'Dishwasher'    : 10,
    'Heat Pump'     : 5,
    'Kitchen Fridge': 2,
    'TV'            : 1,
}

DROP_COLS = [
    'MHE', 'RSE', 'GRE', 'B1E', 'BME', 'EQE', 'OFE', 'UTE',
    'B2E', 'DNE', 'EBE', 'OUE', 'UNE', 'HTE', 'WOE', 'FRE',
]

## 3. Feature Engineering Functions

Same implementation as the preprocessing notebook.

In [3]:
def normalize_with_cap(arr: np.ndarray, cap: float) -> np.ndarray:
    arr = np.clip(arr, 0, None).astype(np.float32)
    if cap <= 0:
        return arr
    return np.clip(arr / cap, 0.0, 1.0).astype(np.float32)


def compute_autocorr(window: np.ndarray) -> np.ndarray:
    n   = len(window)
    w   = window - window.mean()
    var = np.var(w) + 1e-8
    result    = np.zeros(n, dtype=np.float32)
    result[0] = 1.0
    for lag in range(1, n):
        result[lag] = float(np.mean(w[:n - lag] * w[lag:]) / var)
    return np.clip(result, -1.0, 1.0)


def compute_crosscorr(window: np.ndarray, profile: np.ndarray) -> np.ndarray:
    n = min(len(window), len(profile))
    if n == 0:
        return np.zeros(len(window), dtype=np.float32)
    w = window[:n] - window[:n].mean()
    p = profile[:n] - profile[:n].mean()
    denom = float(np.std(w) * np.std(p)) + 1e-8
    corr  = float(np.clip(np.mean(w * p) / denom, -1.0, 1.0))
    return np.full(len(window), corr, dtype=np.float32)


def compute_delta(window: np.ndarray) -> np.ndarray:
    delta = np.diff(window.astype(np.float32), prepend=window[0])
    return np.clip(delta, -1.0, 1.0).astype(np.float32)


def compute_delta2(window: np.ndarray) -> np.ndarray:
    d1 = np.diff(window.astype(np.float32), prepend=window[0])
    d2 = np.diff(d1, prepend=d1[0])
    return np.clip(d2, -1.0, 1.0).astype(np.float32)

## 4. Data Loading & Cleaning

In [4]:
def load_and_clean_ampds(csv_path: str) -> pd.DataFrame:
    """
    Load Electricity_P.csv and apply the same cleaning as the
    preprocessing notebook:
      - Drop unused columns
      - Rename to human-readable names
      - Parse timestamps
      - Clip negatives
      - Apply 50W hard threshold per appliance
    """
    ampd = pd.read_csv(csv_path)
    ampd = ampd.drop(columns=[c for c in DROP_COLS if c in ampd.columns])
    ampd = ampd.rename(columns=APPLIANCE_NAMES)
    ampd = ampd.rename(columns={ampd.columns[0]: 'time'})
    ampd['time'] = pd.to_datetime(ampd['time'], unit='s', errors='coerce')
    ampd = ampd.set_index('time').sort_index()

    # Clip negatives
    for col in ampd.columns:
        ampd[col] = ampd[col].clip(lower=0)

    # Apply 50W hard threshold to appliances only
    for app in TARGET_APPLIANCES:
        if app in ampd.columns:
            ampd[app] = ampd[app].where(ampd[app] >= WATTS_ON_THRESHOLD, other=0.0)

    return ampd

## 5. Event Extraction

In [5]:
def extract_test_events(
    df_test: pd.DataFrame,
    appliance: str,
    norm_caps: dict,
    crosscorr_profile: np.ndarray,
    max_len: int
) -> list:
    """
    Extract all ON events from the test split for one appliance.

    Returns a list of dicts, each representing one event:
        'X'                : np.ndarray (1, max_len, 7)  — model input
        'mask'             : np.ndarray (1, max_len)     — 1=real, 0=padding
        'ground_truth_norm': np.ndarray (real_len,)      — normalised target
        'ground_truth_w'   : np.ndarray (real_len,)      — Watts target
        'real_len'         : int
        'appliance'        : str
    """
    if appliance not in df_test.columns:
        return []

    c         = CONTEXT_CONFIG[appliance]
    lead      = c['lead']
    trail     = c['trail']
    min_dur   = MIN_EVENT_DURATION.get(appliance, 2)

    app_arr   = df_test[appliance].values.astype(np.float64)
    mains_arr = df_test[MAINS_COL].values.astype(np.float64)
    times     = df_test.index
    n         = len(df_test)

    cap_m = norm_caps[MAINS_COL]
    cap_a = norm_caps[appliance]

    # Find ON/OFF transitions
    is_on  = (app_arr > 0).astype(int)
    diff   = np.diff(is_on, prepend=0)
    starts = list(np.where(diff == 1)[0])
    ends   = list(np.where(diff == -1)[0])

    if len(starts) > len(ends):
        ends.append(n - 1)

    events = []

    for s, e in zip(starts, ends):
        duration_min = e - s
        if duration_min < min_dur:
            continue

        i_start = max(0, s - lead)
        i_end   = min(n, e + trail)

        mains_seg = mains_arr[i_start:i_end]
        app_seg   = app_arr[i_start:i_end]
        L         = len(mains_seg)

        if L == 0:
            continue

        # Build 7 channels (identical to preprocessing notebook)
        mains_norm = normalize_with_cap(mains_seg, cap_m)
        app_norm   = normalize_with_cap(app_seg,   cap_a)

        ch0 = mains_norm
        ch1 = compute_autocorr(mains_norm)
        ch2 = compute_crosscorr(mains_norm, crosscorr_profile)

        mid_idx  = i_start + L // 2
        mid_idx  = min(mid_idx, len(times) - 1)
        mid_time = times[mid_idx]
        hour     = mid_time.hour + mid_time.minute / 60.0
        sin_h    = float(np.sin(2 * np.pi * hour / 24.0))
        cos_h    = float(np.cos(2 * np.pi * hour / 24.0))
        ch3      = np.full(L, sin_h, dtype=np.float32)
        ch4      = np.full(L, cos_h, dtype=np.float32)
        ch5      = compute_delta(mains_norm)
        ch6      = compute_delta2(mains_norm)

        agg_7ch = np.stack([ch0, ch1, ch2, ch3, ch4, ch5, ch6], axis=-1)  # (L, 7)

        # Pad to MAX_LEN
        real_len = min(L, max_len)
        X        = np.zeros((1, max_len, 7), dtype=np.float32)
        mask     = np.zeros((1, max_len),    dtype=np.float32)
        X[0, :real_len, :]  = agg_7ch[:real_len]
        mask[0, :real_len]  = 1.0

        events.append({
            'X'                : X,
            'mask'             : mask,
            'ground_truth_norm': app_norm[:real_len],
            'ground_truth_w'   : app_seg[:real_len],
            'real_len'         : real_len,
            'appliance'        : appliance,
        })

    return events

## 6. Main: Load Test Events

Configure paths here, then run the cell to load everything.

In [6]:
# ── Configuration ─────────────────────────────────────────────────────────────
CSV_PATH = 'Electricity_P.csv'
NILM_DIR = 'nilm_datasets'

In [7]:
# ── Step 1: Load norm_caps ────────────────────────────────────────────────────
norm_caps_path = os.path.join(NILM_DIR, 'norm_caps.csv')
norm_caps = pd.read_csv(norm_caps_path).iloc[0].to_dict()
print(f'Loaded norm_caps: {len(norm_caps)} entries')
norm_caps

Loaded norm_caps: 7 entries


{'Whole-House Meter': 5369.0,
 'Clothes Dryer': 5005.57,
 'Clothes Washer': 907.0,
 'Dishwasher': 823.0,
 'Kitchen Fridge': 445.0,
 'Heat Pump': 2484.0,
 'TV': 360.0}

In [8]:
# ── Step 2: Load max_lens ─────────────────────────────────────────────────────
max_lens_path = os.path.join(NILM_DIR, 'max_lens.csv')
max_lens = {k: int(v) for k, v in pd.read_csv(max_lens_path).iloc[0].to_dict().items()}
print(f'Loaded max_lens:')
max_lens

Loaded max_lens:


{'Clothes Washer': 46,
 'Dishwasher': 107,
 'Clothes Dryer': 143,
 'Heat Pump': 503,
 'TV': 302,
 'Kitchen Fridge': 373}

In [9]:
# ── Step 3: Load crosscorr profiles ──────────────────────────────────────────
profiles = {}
for app in TARGET_APPLIANCES:
    safe = app.replace(' ', '_').replace('/', '-')
    path = os.path.join(NILM_DIR, f'{safe}_crosscorr_profile.npy')
    if os.path.exists(path):
        profiles[app] = np.load(path)
        print(f'  ✓ {app:<22}  shape={profiles[app].shape}')
    else:
        print(f'  ✗ {app:<22}  NOT FOUND at {path}')

print(f'\nLoaded {len(profiles)} / {len(TARGET_APPLIANCES)} crosscorr profiles')

  ✓ Clothes Dryer           shape=(99,)
  ✓ Clothes Washer          shape=(23,)
  ✓ Dishwasher              shape=(64,)
  ✓ Kitchen Fridge          shape=(30,)
  ✓ Heat Pump               shape=(148,)
  ✓ TV                      shape=(69,)

Loaded 6 / 6 crosscorr profiles


In [10]:
# ── Step 4: Load and clean AMPds CSV ─────────────────────────────────────────
print(f'Loading {CSV_PATH}...')
ampd = load_and_clean_ampds(CSV_PATH)
print(f'Total rows : {len(ampd):,}')
print(f'Columns    : {list(ampd.columns)}')
ampd.head()

Loading Electricity_P.csv...
Total rows : 1,051,200
Columns    : ['Whole-House Meter', 'Clothes Washer', 'Dishwasher', 'Heat Pump', 'Clothes Dryer', 'Kitchen Fridge', 'TV']


,Whole-House Meter,Clothes Washer,Dishwasher,Heat Pump,Clothes Dryer,Kitchen Fridge,TV
time,,,,,,,
2012-04-01 07:00:00,918,0,0,0,0,0,0
2012-04-01 07:01:00,913,0,0,0,0,0,0
2012-04-01 07:02:00,872,0,0,0,0,0,0
2012-04-01 07:03:00,872,0,0,0,0,0,0
2012-04-01 07:04:00,772,0,0,0,0,0,0


In [11]:
# ── Step 5: Chronological split — take test portion (last 15%) ───────────────
n       = len(ampd)
val_end = int(n * 0.85)
df_test = ampd.iloc[val_end:].copy()

print(f'Full dataset : {n:,} rows')
print(f'Test split   : {len(df_test):,} rows  '
      f'({df_test.index[0].date()} → {df_test.index[-1].date()})')
df_test.head()

Full dataset : 1,051,200 rows
Test split   : 157,680 rows  (2013-12-12 → 2014-04-01)


,Whole-House Meter,Clothes Washer,Dishwasher,Heat Pump,Clothes Dryer,Kitchen Fridge,TV
time,,,,,,,
2013-12-12 19:00:00,895,0,0,0,0,118,0
2013-12-12 19:01:00,799,0,0,0,0,0,0
2013-12-12 19:02:00,795,0,0,0,0,0,0
2013-12-12 19:03:00,653,0,0,0,0,0,0
2013-12-12 19:04:00,658,0,0,0,0,0,0


In [12]:
# ── Step 6: Extract events per appliance ─────────────────────────────────────
all_events = {}

for app in TARGET_APPLIANCES:
    if app not in profiles or app not in max_lens:
        print(f'  Skipping {app} — missing profile or max_len')
        continue

    events = extract_test_events(
        df_test           = df_test,
        appliance         = app,
        norm_caps         = norm_caps,
        crosscorr_profile = profiles[app],
        max_len           = max_lens[app],
    )
    all_events[app] = events
    print(f'  {app:<22}: {len(events):>4} events  (max_len={max_lens[app]})')

total = sum(len(v) for v in all_events.values())
print(f'\nTotal test events across all appliances: {total}')

  Clothes Dryer         :   78 events  (max_len=143)
  Clothes Washer        :  120 events  (max_len=46)
  Dishwasher            :  108 events  (max_len=107)
  Kitchen Fridge        : 4019 events  (max_len=373)
  Heat Pump             :  420 events  (max_len=503)
  TV                    :  210 events  (max_len=302)

Total test events across all appliances: 4955


## 7. Inspect an Event

Quick sanity check on the shape and values of an extracted event.

In [13]:
# Pick the first event from the first available appliance
sample_app = next(app for app, evs in all_events.items() if evs)
sample_ev  = all_events[sample_app][0]

print(f'Appliance       : {sample_ev["appliance"]}')
print(f'X shape         : {sample_ev["X"].shape}   (1, max_len, 7)')
print(f'mask shape      : {sample_ev["mask"].shape}')
print(f'real_len        : {sample_ev["real_len"]}')
print(f'ground_truth_w  : min={sample_ev["ground_truth_w"].min():.1f} W  '
      f'max={sample_ev["ground_truth_w"].max():.1f} W')

Appliance       : Clothes Dryer
X shape         : (1, 143, 7)   (1, max_len, 7)
mask shape      : (1, 143)
real_len        : 109
ground_truth_w  : min=0.0 W  max=4650.0 W


## 8. Run Inference on Test Events

> Requires `nilm_inference.py` and model checkpoints in `checkpoints/`.

In [14]:
import sys
import torch
sys.path.insert(0, '.')
from nilm_inference import NILMBackend

backend = NILMBackend.from_checkpoints(
    checkpoint_dir = 'checkpoints',
    nilm_dir       = NILM_DIR,
)
print('Backend loaded.')

[NILMBackend] norm_caps loaded — mains cap = 5369.0 W
[NILMBackend] max_lens loaded — 6 appliances
[NILMBackend] cross-corr profiles loaded — 6 appliances
  Loaded: Clothes Dryer           max_len= 143  opt_thr=0.7600  (3804.2 W)
  Loaded: Clothes Washer          max_len=  46  opt_thr=0.0500  (45.4 W)
  Loaded: Dishwasher              max_len= 107  opt_thr=0.7250  (596.7 W)
  Loaded: Heat Pump               max_len= 503  opt_thr=0.5050  (1254.4 W)
  Loaded: Kitchen Fridge          max_len= 373  opt_thr=0.2650  (117.9 W)
  Loaded: TV                      max_len= 302  opt_thr=0.2100  (75.6 W)
[NILMBackend] Ready — 6 models loaded on cuda
Backend loaded.


In [16]:
print('--- Running inference on test events ---\n')
results = {}

for app, events in all_events.items():
    if not events:
        continue

    model  = backend.models[app]
    meta   = backend.metas[app]
    device = backend.device
    mae_list = []

    for ev in events:
        X        = ev['X']               # (1, max_len, 7)
        gt_w     = ev['ground_truth_w']  # Watts, for evaluation
        real_len = ev['real_len']

        with torch.no_grad():
            x_tensor  = torch.from_numpy(X).to(device)
            pred_norm = model(x_tensor).cpu().numpy()   # (1, max_len)

        # Trim and denormalise
        pred_w = pred_norm[0, :real_len] * meta.norm_cap  # Watts

        mae = float(np.mean(np.abs(pred_w - gt_w[:real_len])))
        mae_list.append(mae)

    mean_mae = float(np.mean(mae_list))
    results[app] = {'n_events': len(mae_list), 'mean_mae_w': mean_mae}
    print(f'{app:<22}: {len(mae_list):>4} events   Mean MAE = {mean_mae:.2f} W')

print('\nDone.')

--- Running inference on test events ---

Clothes Dryer         :   78 events   Mean MAE = 119.49 W
Clothes Washer        :  120 events   Mean MAE = 112.55 W
Dishwasher            :  108 events   Mean MAE = 101.49 W
Kitchen Fridge        : 4019 events   Mean MAE = 14.75 W
Heat Pump             :  420 events   Mean MAE = 89.73 W
TV                    :  210 events   Mean MAE = 42.24 W

Done.


In [17]:
# Summary table
pd.DataFrame(results).T.rename(columns={'n_events': 'Events', 'mean_mae_w': 'Mean MAE (W)'})

,Events,Mean MAE (W)
Clothes Dryer,78.0,119.490541
Clothes Washer,120.0,112.552989
Dishwasher,108.0,101.490727
Kitchen Fridge,4019.0,14.751851
Heat Pump,420.0,89.727530
TV,210.0,42.240089


In [18]:
output_dir = 'nilm_results'
os.makedirs(output_dir, exist_ok=True)

# Save the summary metrics
summary_df = pd.DataFrame(results).T.rename(
    columns={'n_events': 'Events', 'mean_mae_w': 'Mean MAE (W)'}
)
summary_path = os.path.join(output_dir, 'nilm_test_results_summary.csv')
summary_df.to_csv(summary_path, index=True)

# Optionally save all extracted test events if you want the full dataset
events_path = os.path.join(output_dir, 'nilm_test_events.pkl')
pd.to_pickle(all_events, events_path)

print(f'Saved summary to: {summary_path}')
print(f'Saved events to:  {events_path}')

Saved summary to: nilm_results\nilm_test_results_summary.csv
Saved events to:  nilm_results\nilm_test_events.pkl
